In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv("heart1.csv")
df['Cholesterol'] = df['Cholesterol'].replace(0, np.nan)
df['Cholesterol'] = df['Cholesterol'].fillna(df['Cholesterol'].median())
df_encoded = pd.get_dummies(
    df,
    columns=[
        'Sex',
        'ChestPainType',
        'RestingECG',
        'ExerciseAngina',
        'ST_Slope'
    ],
    drop_first=True
)
from sklearn.model_selection import train_test_split

X = df_encoded.drop("HeartDisease", axis=1)
y = df_encoded["HeartDisease"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=600,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
from sklearn.metrics import accuracy_score

y_pred = rf.predict(X_test)
print("Improved RF Accuracy:", accuracy_score(y_test, y_pred))


Improved RF Accuracy: 0.8532608695652174


In [4]:


import pandas as pd

feature_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

top_features = feature_importance.head(15).index

X_train_sel = X_train[top_features]
X_test_sel = X_test[top_features]

rf.fit(X_train_sel, y_train)
y_pred_sel = rf.predict(X_test_sel)

print("Accuracy after feature selection:",
      accuracy_score(y_test, y_pred_sel))


Accuracy after feature selection: 0.8586956521739131


In [2]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Predictions
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("BASELINE RANDOM FOREST METRICS")
print("--------------------------------")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


BASELINE RANDOM FOREST METRICS
--------------------------------
Accuracy : 0.8532608695652174
Precision: 0.8787878787878788
Recall   : 0.8529411764705882
F1-score : 0.8656716417910447
ROC-AUC  : 0.9271879483500718

Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.85      0.84        82
           1       0.88      0.85      0.87       102

    accuracy                           0.85       184
   macro avg       0.85      0.85      0.85       184
weighted avg       0.85      0.85      0.85       184

Confusion Matrix:

[[70 12]
 [15 87]]
